# LeetCode 2028 — Find Missing Observations bằng Python

Link đề: https://leetcode.com/problems/find-missing-observations/ · Độ khó: **Medium** · Chủ đề: Array, Math, Simulation

Skeleton LeetCode:

```python
class Solution:
    def missingRolls(self, rolls: List[int], mean: int, n: int) -> List[int]:
        ...
```

## 1. Mục tiêu học

1. Chuyển điều kiện "trung bình bằng `mean`" thành điều kiện **tổng**: tổng các lần tung bị mất phải bằng `need = mean·(n+m) − sum(rolls)`.
2. Nhận ra điều kiện **tồn tại**: `n ≤ need ≤ 6n` (mỗi xúc xắc từ 1 tới 6).
3. Xây dựng đáp án bằng **chia đều** (`divmod`) hoặc **tham lam rót dần**.
4. Viết **hàm kiểm tra hợp lệ** cho bài có nhiều đáp án, và đối chiếu với vét cạn.
5. Tránh các lỗi: quên kiểm phạm vi, dùng sai tổng số lần tung, phân phối phần dư làm vượt 6.

## 2. Đề bài & ví dụ

Có `n + m` lần tung xúc xắc 6 mặt (mặt từ 1 tới 6). `n` kết quả bị **mất**, chỉ còn `m` kết quả trong mảng `rolls`. Bạn biết **trung bình** của cả `n + m` lần tung là số nguyên `mean`.

Trả về một mảng độ dài `n` gồm các kết quả bị mất sao cho trung bình của cả `n + m` lần đúng bằng `mean`. Nếu có nhiều đáp án, trả về **bất kỳ**; nếu không tồn tại, trả về **mảng rỗng**.

**Ví dụ 1:** `rolls = [3,2,4,3], mean = 4, n = 2` → `[6,6]` vì `(3+2+4+3+6+6)/6 = 4`.

**Ví dụ 2:** `rolls = [1,5,6], mean = 3, n = 4` → ví dụ `[2,3,2,2]` vì `(1+5+6+2+3+2+2)/7 = 3`.

**Ví dụ 3:** `rolls = [1,2,3,4], mean = 6, n = 4` → `[]` (không thể đạt trung bình 6).

**Ràng buộc:** `1 ≤ n, m ≤ 10⁵`, `1 ≤ rolls[i], mean ≤ 6`.

In [1]:
import random
import time
from itertools import product, combinations_with_replacement
from typing import List

examples = [([3, 2, 4, 3], 4, 2, [6, 6]), ([1, 5, 6], 3, 4, [2, 3, 2, 2]), ([1, 2, 3, 4], 6, 4, [])]

def is_valid(rolls, mean, n, ans):
    """Đáp án hợp lệ: đúng n phần tử trong [1, 6] và trung bình tổng thể bằng mean."""
    return len(ans) == n and all(1 <= x <= 6 for x in ans) and sum(rolls) + sum(ans) == mean * (n + len(rolls))

for rolls, mean, n, exp in examples:
    print(rolls, mean, n, "-> một đáp án:", exp, "| hợp lệ:", is_valid(rolls, mean, n, exp) if exp else "(rỗng)")

[3, 2, 4, 3] 4 2 -> một đáp án: [6, 6] | hợp lệ: True
[1, 5, 6] 3 4 -> một đáp án: [2, 3, 2, 2] | hợp lệ: True
[1, 2, 3, 4] 6 4 -> một đáp án: [] | hợp lệ: (rỗng)


## 3. Kiến thức nền

### 3.1 Từ trung bình sang tổng

Trung bình của `N` số bằng `mean` ⇔ tổng bằng `mean · N`. Ở đây `N = n + m`, nên tổng các lần bị mất là `need = mean·(n + m) − sum(rolls)`. Mọi phép tính đều là số nguyên — không cần số thực.

In [2]:
rolls, mean, n = [1, 5, 6], 3, 4
total = mean * (n + len(rolls))
need = total - sum(rolls)
print("tổng cần của cả n+m lần:", total, "| tổng các lần bị mất:", need)

tổng cần của cả n+m lần: 21 | tổng các lần bị mất: 9


### 3.2 Khoảng giá trị của tổng `n` xúc xắc

Mỗi xúc xắc ∈ `[1, 6]` ⇒ tổng `n` xúc xắc ∈ `[n, 6n]`, và **mọi** số nguyên trong khoảng đó đều đạt được (tăng một xúc xắc thêm 1 mỗi lần).

In [3]:
n = 3
sums = sorted({sum(t) for t in product(range(1, 7), repeat=n)})
print("tổng đạt được với 3 xúc xắc:", sums[0], "...", sums[-1], "| đủ mọi số?", sums == list(range(n, 6 * n + 1)))

tổng đạt được với 3 xúc xắc: 3 ... 18 | đủ mọi số? True


### 3.3 `divmod` — chia lấy thương và dư

`q, r = divmod(need, n)` cho `need = q·n + r`, `0 ≤ r < n`. Cho `r` xúc xắc giá trị `q + 1` và `n − r` xúc xắc giá trị `q` ⇒ tổng đúng `need`, các giá trị chênh nhau tối đa 1.

In [4]:
need, n = 9, 4
q, r = divmod(need, n)
ans = [q + 1] * r + [q] * (n - r)
print(q, r, ans, sum(ans))

2 1 [3, 2, 2, 2] 9


### 3.4 Bài nhiều đáp án: kiểm tra bằng hàm hợp lệ

Vì đề chấp nhận **bất kỳ** đáp án đúng, khi test ta không so với một mảng cố định mà dùng `is_valid`. Còn trường hợp `[]` thì kiểm "trả rỗng **khi và chỉ khi** không tồn tại".

In [5]:
print(is_valid([1, 5, 6], 3, 4, [2, 3, 2, 2]), is_valid([1, 5, 6], 3, 4, [3, 2, 2, 2]), is_valid([1, 5, 6], 3, 4, [1, 1, 1, 6]))

True True True


### 3.5 `need` có thể âm hoặc bằng 0

Nếu các lần tung còn lại đã quá lớn so với `mean`, `need = mean·(n+m) − sum(rolls)` có thể `≤ 0`. Khi đó không xúc xắc nào (giá trị ≥ 1) cứu được ⇒ đáp án rỗng. Điều kiện `need < n` đã bao gồm trường hợp này; nhưng nếu ai đó chỉ kiểm `need > 6n` thì sẽ sai.

In [6]:
rolls, mean, n = [6, 6, 6], 1, 2
need = mean * (n + len(rolls)) - sum(rolls)
print("need =", need, "-> n <= need?", n <= need)

need = -13 -> n <= need? False


## 4. Ý tưởng nền tảng / trực giác

Tính tay ba ví dụ:

| Ví dụ | `sum(rolls)` | `mean·(n+m)` | `need` | khoảng `[n, 6n]` | tồn tại? |
|---|---|---|---|---|---|
| 1 | 12 | 4·6 = 24 | 12 | [2, 12] | ✓ (sát trần: `[6,6]`) |
| 2 | 12 | 3·7 = 21 | 9 | [4, 24] | ✓ |
| 3 | 10 | 6·8 = 48 | 38 | [4, 24] | ✗ (38 > 24) |

Khi tồn tại, chia đều `need` cho `n` xúc xắc: ví dụ 2 `need = 9, n = 4` ⇒ `q = 2, r = 1` ⇒ `[3, 2, 2, 2]`.

In [7]:
for rolls, mean, n, _ in examples:
    need = mean * (n + len(rolls)) - sum(rolls)
    print(f"need={need}, khoảng=[{n}, {6 * n}] -> {'tồn tại' if n <= need <= 6 * n else 'không tồn tại'}")

need=12, khoảng=[2, 12] -> tồn tại
need=9, khoảng=[4, 24] -> tồn tại
need=38, khoảng=[4, 24] -> không tồn tại


## 5. Cách 1 — Brute Force / baseline: thử mọi bộ `n` xúc xắc

**Ý tưởng:** `product(range(1, 7), repeat=n)` sinh `6ⁿ` bộ; trả về bộ đầu tiên có tổng `need`.

In [8]:
def missing_brute(rolls, mean, n):
    need = mean * (n + len(rolls)) - sum(rolls)
    for t in product(range(1, 7), repeat=n):
        if sum(t) == need:
            return list(t)
    return []

for rolls, mean, n, exp in examples:
    got = missing_brute(rolls, mean, n)
    print(got, "| hợp lệ" if got and is_valid(rolls, mean, n, got) else "| rỗng" if not got else "| SAI")

[6, 6] | hợp lệ
[1, 1, 1, 6] | hợp lệ
[] | rỗng


**Độ phức tạp:** `O(6ⁿ · n)` — với `n = 10` đã là `6·10⁷` bộ. Không thể với `n = 10⁵`. Space `O(n)`.

**Nút thắt:** ta không cần **liệt kê** để biết có tồn tại hay không — chỉ cần so `need` với `[n, 6n]`; và việc xây đáp án có công thức trực tiếp.

In [9]:
for n in (4, 5, 6, 7):
    t = time.perf_counter(); missing_brute([1], 6, n)      # không tồn tại -> phải duyệt hết 6^n
    print(f"n={n}: {6 ** n} bộ, {time.perf_counter() - t:.3f}s")

n=4: 1296 bộ, 0.000s
n=5: 7776 bộ, 0.000s
n=6: 46656 bộ, 0.003s
n=7: 279936 bộ, 0.020s


## 6. Ý tưởng thuật toán / bước đột phá

**Insight:** bài toán chỉ phụ thuộc **tổng** `need` của các lần bị mất.

**Quy tắc "nếu… thì…":**
- **Nếu** `need < n` (kể cả `need ≤ 0`) hoặc `need > 6n` **thì** trả `[]`.
- **Ngược lại**, `q, r = divmod(need, n)`; trả `r` số `q + 1` và `n − r` số `q`.

**Vì sao giá trị nằm trong `[1, 6]`:** từ `n ≤ need ≤ 6n` ⇒ `1 ≤ need/n ≤ 6` ⇒ `1 ≤ q ≤ 6`. Nếu `q = 6` thì `need = 6n` ⇒ `r = 0`, không có giá trị `q + 1 = 7`. Vậy mọi phần tử ∈ `[1, 6]`.

**Cách tham lam tương đương:** bắt đầu với `n` số 1 (tổng `n`), còn thiếu `extra = need − n`; rót cho từng xúc xắc tối đa `+5` cho tới khi hết `extra`.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — Duyệt đa tập (`combinations_with_replacement`)

Thứ tự trong đáp án không quan trọng, nên chỉ cần duyệt các **đa tập** `n` phần tử từ `{1..6}`: số lượng `C(n+5, 5)` (đa thức bậc 5), ít hơn nhiều so với `6ⁿ`. Vẫn chỉ hợp với `n` nhỏ.

In [10]:
def missing_multiset(rolls, mean, n):
    need = mean * (n + len(rolls)) - sum(rolls)
    for t in combinations_with_replacement(range(1, 7), n):
        if sum(t) == need:
            return list(t)
    return []

for rolls, mean, n, exp in examples:
    got = missing_multiset(rolls, mean, n)
    assert (got == []) == (exp == []) and (not got or is_valid(rolls, mean, n, got))
print([missing_multiset(r, m, n) for r, m, n, _ in examples])
from math import comb
print("n = 20: 6^n =", 6 ** 20, "| số đa tập C(n+5,5) =", comb(25, 5))

[[6, 6], [1, 1, 1, 6], []]
n = 20: 6^n = 3656158440062976 | số đa tập C(n+5,5) = 53130


**Độ phức tạp Cách 2:** `O(C(n+5, 5) · n)`. Với `n = 20` khoảng `5·10⁴` đa tập — được, nhưng `n = 10⁵` thì không.

### 7.2 Cách 3 — Tham lam: bắt đầu toàn 1, rót phần thiếu

In [11]:
def missing_greedy(rolls, mean, n):
    need = mean * (n + len(rolls)) - sum(rolls)
    if need < n or need > 6 * n:
        return []
    ans = [1] * n
    extra = need - n
    for i in range(n):
        add = min(5, extra)            # mỗi xúc xắc nhận thêm tối đa 5
        ans[i] += add
        extra -= add
        if extra == 0:
            break
    return ans

for rolls, mean, n, exp in examples:
    got = missing_greedy(rolls, mean, n)
    assert (got == []) == (exp == []) and (not got or is_valid(rolls, mean, n, got))
print([missing_greedy(r, m, n) for r, m, n, _ in examples])

[[6, 6], [6, 1, 1, 1], []]


In [12]:
class SolutionGreedy:
    def missingRolls(self, rolls: List[int], mean: int, n: int) -> List[int]:
        need = mean * (n + len(rolls)) - sum(rolls)
        if need < n or need > 6 * n:
            return []
        ans = [1] * n
        extra = need - n
        i = 0
        while extra:
            add = min(5, extra)
            ans[i] += add
            extra -= add
            i += 1
        return ans

print([SolutionGreedy().missingRolls(r, m, n) for r, m, n, _ in examples])

[[6, 6], [6, 1, 1, 1], []]


**Độ phức tạp Cách 3:** `O(n + m)` thời gian (tính `sum(rolls)` mất `O(m)`), `O(n)` cho đáp án.

### 7.3 Cách 4 — Chia đều bằng `divmod` (bản nộp chính)

In [13]:
def missing_divmod(rolls, mean, n):
    need = mean * (n + len(rolls)) - sum(rolls)
    if not (n <= need <= 6 * n):
        return []
    q, r = divmod(need, n)
    return [q + 1] * r + [q] * (n - r)

for rolls, mean, n, exp in examples:
    got = missing_divmod(rolls, mean, n)
    assert (got == []) == (exp == []) and (not got or is_valid(rolls, mean, n, got))
print([missing_divmod(r, m, n) for r, m, n, _ in examples])

[[6, 6], [3, 2, 2, 2], []]


**Độ phức tạp Cách 4:** `O(n + m)`, Space `O(n)` cho đáp án. Ưu điểm: các giá trị "đều" nhất có thể (chênh nhau ≤ 1).

## 8. TRACE từng bước

Trace Cách 3 (tham lam) và Cách 4 trên ví dụ 2 (`need = 9`, `n = 4`):

**Tham lam:** bắt đầu `[1,1,1,1]`, `extra = 5`.

| i | add = min(5, extra) | ans | extra còn |
|---|---|---|---|
| 0 | 5 | [6,1,1,1] | 0 |

**divmod:** `q, r = divmod(9, 4) = (2, 1)` ⇒ `[3, 2, 2, 2]`. Cả hai đều hợp lệ (tổng 9).

In [14]:
def trace_greedy(rolls, mean, n):
    need = mean * (n + len(rolls)) - sum(rolls)
    print(f"sum(rolls)={sum(rolls)}, mean*(n+m)={mean * (n + len(rolls))}, need={need}, khoảng=[{n}, {6 * n}]")
    if need < n or need > 6 * n:
        print("-> không tồn tại"); return []
    ans = [1] * n; extra = need - n
    print("bắt đầu:", ans, "extra =", extra)
    for i in range(n):
        if extra == 0: break
        add = min(5, extra); ans[i] += add; extra -= add
        print(f"i={i}: +{add} -> {ans}, extra={extra}")
    return ans

print("Đáp án:", trace_greedy([1, 5, 6], 3, 4))
print()
print("Đáp án:", trace_greedy([3, 2, 4, 3], 4, 2))
print()
print("Đáp án:", trace_greedy([1, 2, 3, 4], 6, 4))

sum(rolls)=12, mean*(n+m)=21, need=9, khoảng=[4, 24]
bắt đầu: [1, 1, 1, 1] extra = 5
i=0: +5 -> [6, 1, 1, 1], extra=0
Đáp án: [6, 1, 1, 1]

sum(rolls)=12, mean*(n+m)=24, need=12, khoảng=[2, 12]
bắt đầu: [1, 1] extra = 10
i=0: +5 -> [6, 1], extra=5
i=1: +5 -> [6, 6], extra=0
Đáp án: [6, 6]

sum(rolls)=10, mean*(n+m)=48, need=38, khoảng=[4, 24]
-> không tồn tại
Đáp án: []


In [15]:
def trace_divmod(rolls, mean, n):
    need = mean * (n + len(rolls)) - sum(rolls)
    if not (n <= need <= 6 * n):
        print(f"need={need} ngoài [{n}, {6 * n}] -> []"); return []
    q, r = divmod(need, n)
    ans = [q + 1] * r + [q] * (n - r)
    print(f"need={need}, divmod -> q={q}, r={r}: {r} số {q + 1} và {n - r} số {q} -> {ans} (tổng {sum(ans)})")
    return ans

for rolls, mean, n, _ in examples:
    trace_divmod(rolls, mean, n)

need=12, divmod -> q=6, r=0: 0 số 7 và 2 số 6 -> [6, 6] (tổng 12)
need=9, divmod -> q=2, r=1: 1 số 3 và 3 số 2 -> [3, 2, 2, 2] (tổng 9)
need=38 ngoài [4, 24] -> []


## 9. Code nộp LeetCode

In [16]:
class Solution:
    def missingRolls(self, rolls: List[int], mean: int, n: int) -> List[int]:
        need = mean * (n + len(rolls)) - sum(rolls)   # tổng các lần tung bị mất
        if need < n or need > 6 * n:                  # mỗi xúc xắc chỉ từ 1 tới 6
            return []
        q, r = divmod(need, n)
        return [q + 1] * r + [q] * (n - r)            # chia đều, phần dư cho r xúc xắc

for rolls, mean, n, exp in examples:
    got = Solution().missingRolls(rolls, mean, n)
    assert (got == []) == (exp == []) and (not got or is_valid(rolls, mean, n, got))
print("Ví dụ đề: OK", [Solution().missingRolls(r, m, n) for r, m, n, _ in examples])

Ví dụ đề: OK [[6, 6], [3, 2, 2, 2], []]


## 10. Vì sao thuật toán đúng

**Tồn tại ⇔ `n ≤ need ≤ 6n`.**
- (⇒) Nếu có đáp án `x_1..x_n` với `1 ≤ x_i ≤ 6` và `Σx_i = need` thì `n ≤ need ≤ 6n`.
- (⇐) Cách 4 xây được đáp án (chứng minh dưới).

**Đáp án của Cách 4 hợp lệ.** `need = q·n + r` với `0 ≤ r < n`. Tổng `r(q+1) + (n−r)q = qn + r = need` ✓. Từ `n ≤ need` ⇒ `q ≥ 1` (vì nếu `q = 0` thì `need = r < n`). Từ `need ≤ 6n` ⇒ `q ≤ 6`, và nếu `q = 6` thì `need ≥ 6n` ⇒ `need = 6n`, `r = 0` ⇒ không có phần tử `7`. Vậy mọi phần tử ∈ `[1, 6]` ✓. Do đó trung bình tổng thể `= (sum(rolls) + need)/(n+m) = mean` ✓.

**Cách 3 hợp lệ.** Sau vòng lặp, tổng tăng đúng `need − n`; mỗi xúc xắc nhận thêm ≤ 5 ⇒ ≤ 6; vòng lặp đủ chỗ vì `need − n ≤ 5n`.

## 11. Phân tích Time / Space

Gọi `m = len(rolls)`, `n` = số lần bị mất.

| Cách | Time | Space (ngoài đáp án) |
|---|---|---|
| 1. `product` | `O(6ⁿ · n + m)` | `O(n)` |
| 2. Đa tập | `O(C(n+5,5) · n + m)` | `O(n)` |
| 3. Tham lam | `O(n + m)` | `O(1)` |
| 4. `divmod` | `O(n + m)` | `O(1)` |

Phải đọc hết `rolls` để tính tổng và phải xuất `n` số ⇒ `Ω(n + m)` — các Cách 3, 4 là tối ưu.

In [17]:
rng = random.Random(2028)
rolls = [rng.randint(1, 6) for _ in range(10**5)]
for name, fn in [("greedy", missing_greedy), ("divmod", missing_divmod), ("Solution", lambda r, m, n: Solution().missingRolls(r, m, n))]:
    t = time.perf_counter()
    res = [fn(rolls, mean, 10**5) for mean in range(1, 7)]
    ok = all((not a) or is_valid(rolls, mean, 10**5, a) for mean, a in zip(range(1, 7), res))
    print(f"{name:8s}: số mean có đáp án = {sum(1 for a in res if a)}/6, hợp lệ={ok} ({time.perf_counter() - t:.3f}s)")

greedy  : số mean có đáp án = 2/6, hợp lệ=True (0.017s)
divmod  : số mean có đáp án = 2/6, hợp lệ=True (0.011s)
Solution: số mean có đáp án = 2/6, hợp lệ=True (0.011s)


## 12. Bộ test

- Ví dụ đề (bằng `is_valid`).
- Biên: `need` đúng bằng `n` (toàn 1), đúng bằng `6n` (toàn 6), `need = n − 1`, `need = 6n + 1`, `need ≤ 0`.
- Đối chiếu với brute force: với `n ≤ 5`, **tồn tại** theo vét cạn ⇔ các cách trả khác rỗng; và mọi đáp án khác rỗng đều hợp lệ.

In [18]:
fns = {
    "brute": missing_brute, "multiset": missing_multiset, "greedy": missing_greedy, "divmod": missing_divmod,
    "Solution": lambda r, m, n: Solution().missingRolls(r, m, n),
    "SolutionGreedy": lambda r, m, n: SolutionGreedy().missingRolls(r, m, n),
}
def check_all(rolls, mean, n, exists):
    for name, fn in fns.items():
        got = fn(rolls, mean, n)
        assert (got != []) == exists, (name, rolls, mean, n, got)
        assert not got or is_valid(rolls, mean, n, got), (name, rolls, mean, n, got)

for rolls, mean, n, exp in examples:
    check_all(rolls, mean, n, exp != [])
# biên (tự dựng): rolls = [6], mean thay đổi, n = 3 -> need = mean*4 - 6
check_all([6], 3, 3, True)    # need = 6 = 2n
check_all([1], 1, 3, True)    # need = 3 = n (toàn 1)
check_all([6], 6, 3, True)    # need = 18 = 6n (toàn 6)
check_all([2], 1, 3, False)   # need = 2 < n
check_all([1], 6, 1, False)   # need = 11 > 6
check_all([6, 6, 6], 1, 1, False)   # need = 4 - 18 < 0
print("Ví dụ + biên: OK")

Ví dụ + biên: OK


In [19]:
rng = random.Random(42)
pos = neg = 0
for _ in range(600):
    m = rng.randint(1, 6); n = rng.randint(1, 5)
    rolls = [rng.randint(1, 6) for _ in range(m)]; mean = rng.randint(1, 6)
    exists = bool(missing_brute(rolls, mean, n))
    pos += exists; neg += not exists
    check_all(rolls, mean, n, exists)
print(f"600 test ngẫu nhiên ({pos} có đáp án, {neg} không): mọi cách nhất quán với vét cạn")

600 test ngẫu nhiên (251 có đáp án, 349 không): mọi cách nhất quán với vét cạn


## 13. Lỗi người mới hay gặp

### Lỗi 1 — Quên kiểm phạm vi `[n, 6n]`

Không kiểm thì `divmod` vẫn chạy và trả về giá trị vô nghĩa (ví dụ `> 6` hoặc `≤ 0`).

In [20]:
def wrong_no_range(rolls, mean, n):
    need = mean * (n + len(rolls)) - sum(rolls)
    q, r = divmod(need, n)                          # SAI: không kiểm n <= need <= 6n
    return [q + 1] * r + [q] * (n - r)

rolls, mean, n, _ = examples[2]
got = wrong_no_range(rolls, mean, n)
print("sai:", got, "hợp lệ?", is_valid(rolls, mean, n, got), "| đúng: []")

sai: [10, 10, 9, 9] hợp lệ? False | đúng: []


In [21]:
print("sửa:", missing_divmod(rolls, mean, n))

sửa: []


### Lỗi 2 — Dùng sai tổng số lần tung (`mean·n` hoặc `mean·m`)

Trung bình tính trên **cả** `n + m` lần.

In [22]:
def wrong_total(rolls, mean, n):
    need = mean * n - sum(rolls)                    # SAI: phải là mean * (n + m)
    if not (n <= need <= 6 * n): return []
    q, r = divmod(need, n)
    return [q + 1] * r + [q] * (n - r)

rolls, mean, n, _ = examples[1]
got = wrong_total(rolls, mean, n)
print("sai:", got, "| đúng (một đáp án):", missing_divmod(rolls, mean, n))

sai: [] | đúng (một đáp án): [3, 2, 2, 2]


In [23]:
print("sửa: hợp lệ?", is_valid(rolls, mean, n, missing_divmod(rolls, mean, n)))

sửa: hợp lệ? True


### Lỗi 3 — Dồn toàn bộ phần dư vào một xúc xắc

`[q + r] + [q] * (n − 1)` có thể vượt 6: `need = 23, n = 4` ⇒ `q = 5, r = 3` ⇒ `[8, 5, 5, 5]`.

In [24]:
def wrong_remainder(rolls, mean, n):
    need = mean * (n + len(rolls)) - sum(rolls)
    if not (n <= need <= 6 * n): return []
    q, r = divmod(need, n)
    return [q + r] + [q] * (n - 1)                  # SAI: một xúc xắc nhận cả phần dư

rolls, mean, n = [2], 5, 4                          # need = 5*5 - 2 = 23, n = 4 (tự dựng)
got = wrong_remainder(rolls, mean, n)
print("sai:", got, "hợp lệ?", is_valid(rolls, mean, n, got), "| đúng:", missing_divmod(rolls, mean, n))

sai: [8, 5, 5, 5] hợp lệ? False | đúng: [6, 6, 6, 5]


In [25]:
print("sửa: hợp lệ?", is_valid(rolls, mean, n, missing_divmod(rolls, mean, n)))

sửa: hợp lệ? True


### Lỗi 4 — Chỉ dùng `need // n` cho mọi xúc xắc (bỏ mất phần dư)

In [26]:
def wrong_floor_only(rolls, mean, n):
    need = mean * (n + len(rolls)) - sum(rolls)
    if not (n <= need <= 6 * n): return []
    return [need // n] * n                          # SAI: tổng thiếu need % n

rolls, mean, n, _ = examples[1]
got = wrong_floor_only(rolls, mean, n)
print("sai:", got, "tổng =", sum(got), "hợp lệ?", is_valid(rolls, mean, n, got))

sai: [2, 2, 2, 2] tổng = 8 hợp lệ? False


In [27]:
got = missing_divmod(rolls, mean, n)
print("sửa:", got, "tổng =", sum(got), "hợp lệ?", is_valid(rolls, mean, n, got))

sửa: [3, 2, 2, 2] tổng = 9 hợp lệ? True


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- **"Trung bình bằng X"** ⇒ đổi thành **tổng bằng X·N** để làm việc với số nguyên.
- **"Tìm `n` số trong `[a, b]` có tổng `S`"** ⇒ tồn tại ⇔ `a·n ≤ S ≤ b·n`; xây bằng `divmod` hoặc tham lam rót dần.
- Bài trả **bất kỳ** đáp án ⇒ chọn cách xây dựng đơn giản nhất, test bằng hàm hợp lệ.
- Luôn kiểm các trường hợp `need ≤ 0`.
- Đọc kỹ `n` và `m`: `n` là số **bị mất**, `m = len(rolls)`.

## 15. Biến thể của bài

1. **Xúc xắc `k` mặt** (`[1, k]`): điều kiện `n ≤ need ≤ k·n`.
2. **Mỗi xúc xắc có khoảng riêng `[lo_i, hi_i]`:** tồn tại ⇔ `Σlo_i ≤ need ≤ Σhi_i`; xây bằng tham lam từ `lo_i`.
3. **Đáp án nhỏ nhất theo thứ tự từ điển** (bài tập 5).
4. **Đếm số cách** (dãy có thứ tự) để `n` xúc xắc có tổng `need` — DP `O(n · 6n)` (LeetCode 1155 — Number of Dice Rolls With Target Sum).

Minh họa biến thể 2 (khoảng riêng, tự dựng, kiểm bằng vét cạn):

In [28]:
def fill_ranges(ranges, need):
    lo_sum = sum(lo for lo, _ in ranges); hi_sum = sum(hi for _, hi in ranges)
    if not (lo_sum <= need <= hi_sum): return []
    ans = [lo for lo, _ in ranges]; extra = need - lo_sum
    for i, (lo, hi) in enumerate(ranges):
        add = min(hi - lo, extra); ans[i] += add; extra -= add
    return ans

rng = random.Random(2)
for _ in range(400):
    ranges = []
    for _ in range(rng.randint(1, 4)):
        lo = rng.randint(0, 4); ranges.append((lo, lo + rng.randint(0, 3)))
    need = rng.randint(-1, 25)
    exists = any(sum(t) == need for t in product(*[range(lo, hi + 1) for lo, hi in ranges]))
    got = fill_ranges(ranges, need)
    assert bool(got) == exists
    assert not got or (sum(got) == need and all(lo <= x <= hi for x, (lo, hi) in zip(got, ranges)))
print("Biến thể khoảng riêng khớp vét cạn; ví dụ:", fill_ranges([(1, 3), (2, 2), (0, 5)], 7))

Biến thể khoảng riêng khớp vét cạn; ví dụ: [3, 2, 2]


Minh họa biến thể 4 — **đếm số dãy** `n` xúc xắc (có thứ tự) có tổng `need` (LeetCode 1155 với `k = 6`). Gọi `ways[i][s]` = số cách để `i` xúc xắc đầu có tổng `s`: `ways[i][s] = Σ_{v=1..6} ways[i−1][s−v]`. Nhờ đó ta còn biết bài gốc có **bao nhiêu** đáp án khác nhau (tính theo thứ tự). Kiểm bằng vét cạn `product`:

In [29]:
def count_sequences(n, need, faces=6):
    ways = [1] + [0] * need                  # 0 xúc xắc: chỉ tổng 0
    for _ in range(n):
        new = [0] * (need + 1)
        for s_ in range(need + 1):
            if ways[s_]:
                for v in range(1, faces + 1):
                    if s_ + v <= need:
                        new[s_ + v] += ways[s_]
        ways = new
    return ways[need] if need >= 0 else 0

for n_ in range(1, 6):
    for need_ in range(0, 6 * n_ + 2):
        assert count_sequences(n_, need_) == sum(1 for t in product(range(1, 7), repeat=n_) if sum(t) == need_)
rolls, mean, n = examples[1][:3]
need = mean * (n + len(rolls)) - sum(rolls)
print("Ví dụ 2 có", count_sequences(n, need), "đáp án (dãy có thứ tự) khác nhau; ví dụ 1 có", count_sequences(2, 12))

Ví dụ 2 có 56 đáp án (dãy có thứ tự) khác nhau; ví dụ 1 có 1


Hệ quả thú vị: số đáp án lớn nhất khi `need` ở **giữa** khoảng `[n, 6n]` (phân phối tổng xúc xắc có dạng chuông), và chỉ có **1** đáp án khi `need = n` hoặc `need = 6n`.

In [30]:
n_ = 4
dist = [count_sequences(n_, t) for t in range(n_, 6 * n_ + 1)]
peak = n_ + dist.index(max(dist))
print("4 xúc xắc: số cách theo tổng", list(zip(range(n_, 6 * n_ + 1), dist))[:6], "...")
print("đỉnh tại tổng", peak, "với", max(dist), "cách; hai biên:", dist[0], dist[-1])

4 xúc xắc: số cách theo tổng [(4, 1), (5, 4), (6, 10), (7, 20), (8, 35), (9, 56)] ...
đỉnh tại tổng 14 với 146 cách; hai biên: 1 1


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. `product` | thử `6ⁿ` bộ | `O(6ⁿ·n)` | `O(n)` | kiểm tra, `n ≤ 6` |
| 2. Đa tập | `combinations_with_replacement` | `O(n⁵·n)` | `O(n)` | `n` nhỏ-vừa |
| 3. Tham lam | toàn 1, rót tối đa +5 | `O(n + m)` | `O(1)` | khi có khoảng riêng mỗi phần tử |
| 4. `divmod` | chia đều | `O(n + m)` | `O(1)` | **nộp bài** |

## 17. Bài tập (từ dễ đến khó)

1. **Đoán output:** `missingRolls([6,6,6,6], 6, 3)` và `missingRolls([1,1], 1, 5)` (theo Cách 4).
2. **Tính tay:** `rolls = [4,5,6], mean = 2, n = 3` — có tồn tại không?
3. **Điền TODO:** `need_range(rolls, mean, n)` trả về `(need, lo, hi)` và `True/False` tồn tại.
4. **Tự viết lại:** viết lại Cách 4 không nhìn code.
5. **Mini challenge:** `lexi_smallest(rolls, mean, n)` — đáp án **nhỏ nhất theo thứ tự từ điển** (hoặc `[]`). Gợi ý: đặt mỗi vị trí giá trị nhỏ nhất sao cho phần còn lại vẫn khả thi. Kiểm bằng vét cạn (`product` sinh theo thứ tự từ điển!).

In [31]:
# Bài 3 — TODO (chỉ định nghĩa, không gọi)
def need_range(rolls, mean, n):
    # TODO
    pass

In [32]:
# Bài 4 — TODO (chỉ định nghĩa, không gọi)
def my_missing_rolls(rolls, mean, n):
    # TODO
    pass

In [33]:
# Bài 5 — TODO (chỉ định nghĩa, không gọi)
def lexi_smallest(rolls, mean, n):
    # TODO
    pass

## 18. Gợi ý 3 mức

**Bài 1.** (1) `need = 6·7 − 24 = 18 = 6·3` ⇒ `[6,6,6]`. (2) `need = 1·7 − 2 = 5 = n` ⇒ `[1,1,1,1,1]`. (3) Cả hai đều ở biên.

**Bài 2.** (1) `need = 2·6 − 15 = −3`. (2) `−3 < n = 3`. (3) Không tồn tại ⇒ `[]`.

**Bài 3.** (1) `need = mean·(n + len(rolls)) − sum(rolls)`. (2) `lo, hi = n, 6n`. (3) `lo <= need <= hi`.

**Bài 4.** (1) Kiểm phạm vi. (2) `divmod(need, n)`. (3) `[q+1]*r + [q]*(n−r)`.

**Bài 5.** (1) Vị trí `i` còn `rest = n − i − 1` xúc xắc sau nó. (2) Giá trị nhỏ nhất `v` thỏa `need − v ≤ 6·rest` là `v = max(1, need − 6·rest)`. (3) Kết quả là: các số 1 ở đầu, một số "chuyển tiếp", rồi các số 6 ở cuối.

## 19. Đáp án tham khảo

In [34]:
# Bài 1, 2
assert Solution().missingRolls([6, 6, 6, 6], 6, 3) == [6, 6, 6]
assert Solution().missingRolls([1, 1], 1, 5) == [1, 1, 1, 1, 1]
assert Solution().missingRolls([4, 5, 6], 2, 3) == []
print("Bài 1: [6,6,6] và [1]*5 | Bài 2: []")

Bài 1: [6,6,6] và [1]*5 | Bài 2: []


In [35]:
# Bài 3, 4
def need_range_ans(rolls, mean, n):
    need = mean * (n + len(rolls)) - sum(rolls)
    return need, n, 6 * n, n <= need <= 6 * n

my_missing_rolls_ans = missing_divmod
rng = random.Random(3)
for _ in range(300):
    rolls = [rng.randint(1, 6) for _ in range(rng.randint(1, 5))]; mean = rng.randint(1, 6); n = rng.randint(1, 4)
    *_, ok = need_range_ans(rolls, mean, n)
    got = my_missing_rolls_ans(rolls, mean, n)
    assert ok == bool(missing_brute(rolls, mean, n)) == bool(got)
print("Bài 3, 4: OK; ví dụ 2 ->", need_range_ans([1, 5, 6], 3, 4))

Bài 3, 4: OK; ví dụ 2 -> (9, 4, 24, True)


In [36]:
# Bài 5
def lexi_smallest_ans(rolls, mean, n):
    need = mean * (n + len(rolls)) - sum(rolls)
    if not (n <= need <= 6 * n): return []
    ans = []
    for i in range(n):
        rest = n - i - 1
        v = max(1, need - 6 * rest)                 # nhỏ nhất mà phần còn lại vẫn đủ lớn
        ans.append(v); need -= v
    return ans

rng = random.Random(5)
for _ in range(400):
    rolls = [rng.randint(1, 6) for _ in range(rng.randint(1, 5))]; mean = rng.randint(1, 6); n = rng.randint(1, 5)
    assert lexi_smallest_ans(rolls, mean, n) == missing_brute(rolls, mean, n)   # product sinh theo thứ tự từ điển
print("Bài 5: OK; ví dụ 2 ->", lexi_smallest_ans([1, 5, 6], 3, 4), "| ví dụ 1 ->", lexi_smallest_ans([3, 2, 4, 3], 4, 2))

Bài 5: OK; ví dụ 2 -> [1, 1, 1, 6] | ví dụ 1 -> [6, 6]


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template "n số trong [a, b] có tổng S":**

```python
if not (a * n <= S <= b * n): return []
q, r = divmod(S, n)            # với a = 1, b = 6 ở bài này
return [q + 1] * r + [q] * (n - r)
```

In [37]:
def template(rolls, mean, n):
    S = mean * (n + len(rolls)) - sum(rolls)
    if not (n <= S <= 6 * n): return []
    q, r = divmod(S, n)
    return [q + 1] * r + [q] * (n - r)
print([template(r, m, n) for r, m, n, _ in examples])

[[6, 6], [3, 2, 2, 2], []]


**Checklist:**

- [ ] `need = mean·(n + m) − sum(rolls)` (dùng **cả** `n + m`).
- [ ] Kiểm `n ≤ need ≤ 6n` trước khi xây đáp án (gồm cả `need ≤ 0`).
- [ ] Phân phần dư `r` cho **`r` xúc xắc khác nhau**, mỗi cái +1.
- [ ] Mọi phần tử ∈ `[1, 6]`, độ dài đúng `n`.
- [ ] Trả `[]` khi không tồn tại.
- [ ] Test bằng hàm hợp lệ vì có nhiều đáp án.